# Introduction to Data Pipeline Components

A **data pipeline** is a series of processes that move data from source to destination, making it **ready for analysis, machine learning, or reporting**.  
A typical pipeline consists of three main components: **Ingestion**, **Transformation**, and **Storage**.

---

## 1. Data Ingestion

**Definition:**  
Data ingestion is the process of **collecting and importing raw data** from various sources into a pipeline for further processing.  

**Key Points:**

- Can be **batch** (periodic bulk loads) or **streaming** (continuous real-time feed)  
- Ensures **data availability** for downstream processes  
- Handles **heterogeneous sources**, including structured, semi-structured, or unstructured data  

**Examples of Sources and Tools:**

| Source Type          | Tools / Techniques                          |
| ------------------- | ------------------------------------------- |
| Files (CSV, Excel)   | `pandas.read_csv()`, `pandas.read_excel()` |
| Databases (SQL/NoSQL)| `SQLAlchemy`, `pymongo`, `psycopg2`       |
| APIs / Web Services  | `requests`, `BeautifulSoup`, `Scrapy`     |
| Streaming Data       | `Kafka`, `Spark Streaming`                 |

---

## 2. Data Transformation

**Definition:**  
Data transformation is the process of **cleaning, modifying, and enriching raw data** into a structured format suitable for analysis or modeling.

**Key Steps:**

1. **Cleaning**  
   - Handle missing or inconsistent data  
   - Remove duplicates  
   - Filter invalid or corrupted entries  

2. **Normalization / Scaling**  
   - Standardization, Min-Max Scaling, Robust Scaling  

3. **Feature Engineering**  
   - Create new features from existing data  
   - Extract time-based or derived features  

4. **Encoding Categorical Variables**  
   - One-hot encoding, label encoding  

5. **Aggregation / Summarization**  
   - Summarize or group data for time-based or category-based analysis  


## 3. Data Storage

**Definition:**  
Data storage is the process of **persisting processed data** in a format and location that enables efficient retrieval, querying, and analysis.

**Common Storage Solutions:**

| Storage Type       | Purpose                                      | Tools / Formats              |
| ------------------ | ------------------------------------------- | ---------------------------- |
| Relational DB      | Structured data with defined relationships  | PostgreSQL, MySQL, SQL Server |
| NoSQL / Document   | Semi-structured or unstructured data        | MongoDB, Cassandra, DynamoDB |
| Data Warehouse     | Analytics, reporting, and BI applications  | Snowflake, BigQuery, Redshift |
| Data Lake          | Large-scale raw and processed data storage | S3, HDFS, Azure Data Lake    |
| File-based Storage | Intermediate results, backups, or archives | Parquet, CSV, JSON, HDF5     |

---

## Pipeline Workflow Overview


```
Raw Data → Ingestion → Transformation → Storage → Analysis/ML
```

Each component plays a critical role in ensuring data quality, accessibility, and usability throughout the entire data lifecycle.


In [16]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
import sqlite3

In [17]:
# Create a sample Data
data = {
    "id": range(1, 11),
    "name": ["John", "Mary", "Bob", "Lisa", "Tom", "Alice", "Steve", "Kate", "Paul", "Nina"],
    "age": [28, 35, np.nan, 42, 30, 27, 150, 31, 29, 36],  # 150 is an outlier, Row 3 has NaN
    "gender": ["Male", "Female", "Male", "Female", "Male", "Female", "Male", "Female", "Male", "Female"],
    "city": ["New York", "Los Angeles", "Chicago", "New York", np.nan, "Chicago", "Los Angeles", "New York", "Chicago", "Los Angeles"],
    "income": [50000, 60000, 55000, 70000, 48000, 52000, 65000, 58000, 200000, 63000],  # 200000 is an outlier
    "balance": [2000, 3000, np.nan, 4000, 2200, 2500, 3500, 2800, 2100, 3300]             # Row 3 has NaN
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv("data/raw_data/raw_users.csv", index=False)


### Features of This Dataset:

| Column    | Notes                                    |
| --------- | ---------------------------------------- |
| `age`     | Row 3 has NaN, Row 7 is an outlier (150) |
| `gender`  | Categorical                              |
| `city`    | Row 5 has NaN                            |
| `income`  | Row 9 is an extreme outlier (200000)     |
| `balance` | Row 3 has NaN                            |


In [18]:
# Ingestion Component - Reads data from a CSV file and returns a DataFrame.
def ingest_data(file_path: str) -> pd.DataFrame:
    """Ingest data from a CSV file."""
    return pd.read_csv(file_path)

In [19]:
# Transformation Component - Transforms the raw DataFrame by handling missing values, outliers, and encoding.
def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """Transform the raw DataFrame by handling missing values, outliers, and encoding."""
    # Handle missing values
    imputer = SimpleImputer(strategy='median')
    df['age'] = imputer.fit_transform(df[['age']])
    df['balance'] = imputer.fit_transform(df[['balance']])
    
    # Handle outliers (simple capping)
    df['age'] = df['age'].clip(0, 100)
    df['income'] = df['income'].clip(0, 100000)
    
    # Encode categorical variables
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_cols = encoder.fit_transform(df[['gender', 'city']])
    encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(['gender', 'city']))  
    df = pd.concat([df.drop(['gender', 'city'], axis=1), encoded_df], axis=1)
    return df 
    

In [20]:
# Storage Component - Stores the transformed DataFrame to CSV, Parquet, and SQLite.
def store_data(df: pd.DataFrame, file_path: str) :
    """Store the transformed DataFrame to a CSV file."""
    # Store to CSV
    csv_path = f'{file_path}.csv'
    df.to_csv(csv_path, index=False)
    
    # Store to Parquet
    parquet_path = f'{file_path}.parquet'
    df.to_parquet(parquet_path, index=False)
    
    # Store to SQLite
    conn = sqlite3.connect(f'{file_path}.db')
    df.to_sql('users', conn, if_exists='replace', index=False)
    conn.close()   
    

In [21]:
# Execute the complete data pipeline: Ingestion → Transformation → Storage

# Step 1: Ingest data from CSV
raw_df = ingest_data("data/raw_data/raw_users.csv")
print("Data Ingestion Complete")
print(f"  Shape: {raw_df.shape}")
print(raw_df.head())

# Step 2: Transform the data
transformed_df = transform_data(raw_df)
print("\nData Transformation Complete")
print(f" Shape: {transformed_df.shape}")
print(transformed_df.head())

# Step 3: Store the transformed data
store_data(transformed_df, "data/component_processed_data/processed_users")
print("\nData Storage Complete")
print("  Files saved: processed_users.csv, processed_users.parquet, processed_users.db")

Data Ingestion Complete
  Shape: (10, 7)
   id  name   age  gender         city  income  balance
0   1  John  28.0    Male     New York   50000   2000.0
1   2  Mary  35.0  Female  Los Angeles   60000   3000.0
2   3   Bob   NaN    Male      Chicago   55000      NaN
3   4  Lisa  42.0  Female     New York   70000   4000.0
4   5   Tom  30.0    Male          NaN   48000   2200.0

Data Transformation Complete
 Shape: (10, 11)
   id  name   age  income  balance  gender_Female  gender_Male  city_Chicago  \
0   1  John  28.0   50000   2000.0            0.0          1.0           0.0   
1   2  Mary  35.0   60000   3000.0            1.0          0.0           0.0   
2   3   Bob  31.0   55000   2800.0            0.0          1.0           1.0   
3   4  Lisa  42.0   70000   4000.0            1.0          0.0           0.0   
4   5   Tom  30.0   48000   2200.0            0.0          1.0           0.0   

   city_Los Angeles  city_New York  city_nan  
0               0.0            1.0       0.0  
1